# SmolVLM-256M: Image Preprocessing on the GPU

This notebook tests image preprocessing on the GPU. Earlier tests showed that tile and normalize take about 308 ms on the CPU. This is the largest part of the request before the model.

The script compares three options on the same image: `slow` (the original image processor, on the CPU), `fast-cpu` (the fast image processor, on the CPU) and `fast-gpu` (the fast image processor, on the GPU).

For each option, the script prints the image processor class, the time for tile and normalize, and the time for the full input preparation (JPEG decode, tile, normalize, tokenize and copy to the GPU). It also prints the peak VRAM, the difference of the pixel values from the first option, and whether the model gives the same answer.

DALI and VPI are not tested.

In [ ]:
REPO = "./edge-ai-smol-lvm-250-optimization"
!python {REPO}/gpu_preprocess.py --image {REPO}/frame.jpg

## Result

The test ran on a Colab GPU with one image. The input has 875 tokens (about 13 tiles).

| Option | Image processor | Tile + normalize (ms) | Total (ms) | Speed-up (total) | Peak VRAM (MB) | Pixel diff max / mean | Same answer |
|---|---|---|---|---|---|---|---|
| slow | Idefics3ImageProcessorPil | 286.9 | 299.1 | 1x | 622 | 0.000 / 0.0000 | True |
| fast-cpu | Idefics3ImageProcessor | 101.1 | 117.4 | 2.5x | 689 | 0.204 / 0.0052 | True |
| fast-gpu | Idefics3ImageProcessor | 10.3 | 11.6 | 26x | 768 | 0.204 / 0.0053 | True |

Total is the full input preparation: JPEG decode, tile, normalize, tokenize and copy to the GPU.

- `fast-gpu` takes 11.6 ms for the full input preparation. `slow` takes 299 ms. This saves about 287 ms for each request.
- The fast processor on the CPU is 2.5 times faster than `slow`. The GPU makes it 10 times faster again.
- The fast processors resize with BICUBIC and not LANCZOS (see the warning in the output). The pixel difference is small: 0.005 on average and 0.2 at most. Both fast options have the same difference, so the GPU adds no extra error.
- The model gives the same answer for all options. The test used one image only. Test more images before you use `fast-gpu`.
- `fast-gpu` uses 146 MB more peak VRAM than `slow` (768 MB and 622 MB).
- The earlier tests measured about 308 ms for tile and normalize. This was probably the `slow` (PIL) processor. With `fast-gpu`, preprocessing falls from about 29% to about 1.5% of the request (estimate).